# Example 01 — Duffing Oscillator

Frequency-response analysis of a 1-DOF Duffing oscillator using both Harmonic Balance (HB) and Shooting arc-length continuation, demonstrating the hardening jump phenomenon. (Krack & Gross 2019, §5.1)

In [ ]:
# --- Imports ---
import sys
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Ensure repo src is importable
_repo_root = Path('..').resolve()
if str(_repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(_repo_root / 'src'))

from nlvib.nonlinearities.elements import cubic_spring
from nlvib.systems.oscillators import SingleMassOscillator
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.solvers.shooting import shooting_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions
from nlvib.visualization.plots import plot_harmonic_content, plot_time_series, _split_by_stability

In [ ]:
# --- System setup — parameters from MATLAB NLvib/EXAMPLES/01_Duffing/Duffing.m ---
# mu=1, zeta=0.05, kappa=1, gamma=0.1, P=0.18, H=7, Om_s=0.5, Om_e=1.6
# (Krack & Gross 2019, §5.1)
MASS = 1.0
DAMPING = 0.05          # zeta in MATLAB (was 0.02 — corrected to match reference)
STIFFNESS = 1.0
K3 = 0.1                # gamma in MATLAB (was 0.5 — corrected to match reference)
FORCE_AMPLITUDE = 0.18  # P in MATLAB (was 0.1 — corrected to match reference)
OMEGA_MIN = 0.5
OMEGA_MAX = 1.6         # Om_e in MATLAB (was 1.5 — corrected to match reference)
N_HARMONICS = 7         # H in MATLAB (was 5 — corrected to match reference)

EXCITATION = {'dof': 0, 'amplitude': FORCE_AMPLITUDE, 'harmonic': 1}

system = SingleMassOscillator(m=MASS, d=DAMPING, k=STIFFNESS)
system.add_nonlinear_element(cubic_spring(k3=K3, dof_index=0))
print(f'System: {system}')
print(f'Parameters: m={MASS}, d={DAMPING}, k={STIFFNESS}, k3={K3}, F={FORCE_AMPLITUDE}, H={N_HARMONICS}')
print(f'Omega range: [{OMEGA_MIN}, {OMEGA_MAX}] rad/s')

In [ ]:
# --- HB arc-length continuation ---
n_dof = system.n_dof
omega0 = OMEGA_MIN
n_total = n_dof * (2 * N_HARMONICS + 1)
Q0 = np.zeros(n_total)
denom = abs(-(omega0**2) * MASS + STIFFNESS + 1j * omega0 * DAMPING)
Q0[n_dof] = FORCE_AMPLITUDE / denom if denom > 1e-12 else 0.0

def hb_res_fn(Q, lam):
    return hb_residual(Q, lam, system, N_HARMONICS, EXCITATION)

opts = ContinuationOptions(
    ds_initial=0.02, ds_min=1e-5, ds_max=0.1, max_steps=1000,
    max_newton_iter=20, newton_tol=1e-10, adapt_step=True,
    lambda_min=OMEGA_MIN - 0.05, lambda_max=OMEGA_MAX + 0.05,
)
result_hb = ContinuationSolver().run(hb_res_fn, Q0, omega0, opts)
print(f'HB: {result_hb.n_steps} steps, {result_hb.message}')

omega_hb = result_hb.solutions[:, -1]
Q_arr = result_hb.solutions[:, :-1]  # shape: (n_steps, 2*H+1) for n_dof=1

# Amplitude metric: a_rms = sqrt(sum(Q^2)) / sqrt(2) — matches MATLAB save_data.m
# Uses ALL Fourier coefficients (DC + all harmonics), not just harmonic-1 amplitude.
# This is consistent with the comparison notebook and MATLAB reference.
amp_hb = np.sqrt(np.sum(Q_arr**2, axis=1)) / np.sqrt(2)
stab_hb = ~result_hb.stability

mask_hb = (omega_hb >= OMEGA_MIN) & (omega_hb <= OMEGA_MAX)
print(f'Steps in range [{OMEGA_MIN}, {OMEGA_MAX}]: {mask_hb.sum()}')
print(f'Peak a_rms: {amp_hb[mask_hb].max():.6f} at omega={omega_hb[mask_hb][np.argmax(amp_hb[mask_hb])]:.4f} rad/s')

In [ ]:
# --- Shooting arc-length continuation ---
denom_sh = abs(-(omega0**2) * MASS + STIFFNESS + 1j * omega0 * DAMPING)
lin_amp = FORCE_AMPLITUDE / denom_sh if denom_sh > 1e-12 else 0.0
y0 = np.array([lin_amp, 0.0])

def sh_res_fn(y, lam):
    def _f(t):
        return np.array([FORCE_AMPLITUDE * np.cos(lam * t)])
    return shooting_residual(y, lam, system, n_periods=1, n_steps=128, f_ext_fn=_f)

opts_sh = ContinuationOptions(
    ds_initial=0.02, ds_min=1e-5, ds_max=0.1, max_steps=800,
    max_newton_iter=25, newton_tol=1e-8, adapt_step=True,
    lambda_min=OMEGA_MIN - 0.05, lambda_max=OMEGA_MAX + 0.05,
)
result_sh = ContinuationSolver().run(sh_res_fn, y0, omega0, opts_sh)
print(f'Shooting: {result_sh.n_steps} steps, {result_sh.message}')

omega_sh = result_sh.solutions[:, -1]
# Shooting amplitude: |x(0)| (initial displacement magnitude)
amp_sh = np.abs(result_sh.solutions[:, 0])
stab_sh = ~result_sh.stability

In [ ]:
# --- Plot and save FRF ---
# Axis conventions match comparison notebook (notebooks/comparison/01_duffing.ipynb):
#   x-label: 'Excitation frequency (rad/s)'
#   y-label: 'Response amplitude a_rms'
#   x-limits: [OMEGA_MIN, OMEGA_MAX] = [0.5, 1.6]
#   scale: linear (both axes)

output_dir = Path('..') / 'examples' / '01_duffing' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)

fig_frf, ax_frf = plt.subplots(figsize=(7, 4))

# HB — split by stability
for x_seg, y_seg, is_stable in _split_by_stability(omega_hb, amp_hb, stab_hb):
    ls = '-' if is_stable else '--'
    color = 'tab:blue'
    lbl = ('HB (stable)' if is_stable else 'HB (unstable)') if (
        not any(l.get_label().startswith('HB') for l in ax_frf.get_lines())
    ) else None
    ax_frf.plot(x_seg, y_seg, ls, color=color, label=lbl, linewidth=2)

# Shooting overlay
stable_added = unstable_added = False
for x_seg, y_seg, is_stable in _split_by_stability(omega_sh, amp_sh, stab_sh):
    ls = '-' if is_stable else '--'
    lbl = None
    if is_stable and not stable_added:
        lbl = 'Shooting (stable)'; stable_added = True
    elif not is_stable and not unstable_added:
        lbl = 'Shooting (unstable)'; unstable_added = True
    ax_frf.plot(x_seg, y_seg, ls, color='tab:orange', label=lbl, alpha=0.8)

ax_frf.set_xlabel('Excitation frequency (rad/s)')
ax_frf.set_ylabel('Response amplitude a_rms')
ax_frf.set_title('Duffing Oscillator — FRF (HB + Shooting)')
ax_frf.set_xlim(OMEGA_MIN, OMEGA_MAX)
ax_frf.grid(True, linestyle='--', alpha=0.5)
ax_frf.legend(fontsize=8)
fig_frf.tight_layout()
fig_frf.savefig(output_dir / 'frf.png', dpi=150, bbox_inches='tight')
print('Saved frf.png')

In [ ]:
# --- Harmonic content and time series at peak ---
# Find peak within the valid frequency range
peak_idx_hb = int(np.argmax(amp_hb[mask_hb]))
# Map back to full array index
peak_full_idx = np.where(mask_hb)[0][peak_idx_hb]
omega_res = float(omega_hb[peak_full_idx])
Q_peak_row = Q_arr[peak_full_idx]  # shape: (2*H+1,) for n_dof=1

print(f'Peak frequency: {omega_res:.4f} rad/s')
print(f'Peak a_rms:     {amp_hb[peak_full_idx]:.6f}')

# Extract per-harmonic amplitudes: |Q_k| = sqrt(Q_cos_k^2 + Q_sin_k^2)
# Q layout for n_dof=1: [Q_dc, Q_cos1, Q_sin1, Q_cos2, Q_sin2, ..., Q_cosH, Q_sinH]
harmonic_amps = np.zeros(N_HARMONICS)
for h in range(1, N_HARMONICS + 1):
    harmonic_amps[h-1] = np.sqrt(Q_peak_row[2*h-1]**2 + Q_peak_row[2*h]**2)

# Time series reconstruction from HB coefficients
T = 2.0 * np.pi / omega_res
t_ts = np.linspace(0, T, 256, endpoint=False)
q_ts = np.full_like(t_ts, Q_peak_row[0])  # DC term
dq_ts = np.zeros_like(t_ts)
for h in range(1, N_HARMONICS + 1):
    hw = h * omega_res
    Qc, Qs = Q_peak_row[2*h-1], Q_peak_row[2*h]
    q_ts  += Qc * np.cos(hw * t_ts) + Qs * np.sin(hw * t_ts)
    dq_ts += -hw * Qc * np.sin(hw * t_ts) + hw * Qs * np.cos(hw * t_ts)

fig_hc = plot_harmonic_content(harmonic_amps, omega_res)
fig_hc.savefig(output_dir / 'harmonic_content.png', dpi=150, bbox_inches='tight')
print('Saved harmonic_content.png')

fig_ts = plot_time_series(t_ts, q_ts, dq=dq_ts, dof=0)
fig_ts.axes[0].set_title(f'Time Series at Peak (\u03c9 = {omega_res:.4f} rad/s)')
fig_ts.savefig(output_dir / 'time_series.png', dpi=150, bbox_inches='tight')
print('Saved time_series.png')

In [ ]:
# --- Display FRF inline ---
fig_frf

In [ ]:
# --- Display harmonic content inline ---
fig_hc

In [ ]:
# --- Display time series inline ---
fig_ts

In [ ]:
# --- Summary ---
peak_amp_hb = float(amp_hb[peak_full_idx])
print('=' * 60)
print('SUMMARY — Example 01: Duffing Oscillator')
print('=' * 60)
print(f'  Parameters (matching MATLAB Duffing.m):')
print(f'  {"m":>6} = {MASS},  d = {DAMPING},  k = {STIFFNESS}')
print(f'  {"k3":>6} = {K3},  F = {FORCE_AMPLITUDE},  H = {N_HARMONICS}')
print(f'  Omega range: [{OMEGA_MIN}, {OMEGA_MAX}] rad/s')
print('-' * 60)
print(f'{"Peak a_rms (HB)":<35} {peak_amp_hb:>15.6f}')
print(f'{"Resonance frequency (HB) [rad/s]":<35} {omega_res:>15.6f}')
print(f'{"HB continuation points":<35} {len(omega_hb):>15d}')
print(f'{"  (in frequency range)":<35} {mask_hb.sum():>15d}')
print(f'{"Shooting continuation points":<35} {len(omega_sh):>15d}')
print('-' * 60)
print('Harmonic amplitudes at peak:')
for h_idx, h_amp in enumerate(harmonic_amps):
    print(f'  H{h_idx+1:<33} {h_amp:>15.6e}')
print('=' * 60)
# Amplitude metric: a_rms = sqrt(sum(Q^2)) / sqrt(2) — all harmonics (MATLAB-compatible)